[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/latincy/latincy-book/blob/main/quickstart.ipynb)

This quickstart introduces LatinCy through the opening sentences of Ritchie's *Fabulae Faciles* (1884), a Latin mythology reader widely used in beginning Latin courses. In a few minutes you will install the library, load a model, annotate a text, and run a lemma-based search to find all grammatical forms of a Latin word across a passage — useful for close reading, concordance work, and textual analysis.

The examples use the small model (`la_core_web_sm`). No prior NLP experience is assumed.

## Setup

The cell below installs spaCy and the LatinCy small model automatically when run in Google Colab. If you are working in a local environment, see the [Installing LatinCy models](2_install.ipynb) chapter for installation instructions.

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install -U spacy -q
    !pip install -q "la-core-web-sm @ https://huggingface.co/latincy/la_core_web_sm/resolve/main/la_core_web_sm-3.9.4-py3-none-any.whl"

In [2]:
import spacy

nlp = spacy.load('la_core_web_sm')
print(f"Loaded pipeline: {nlp.meta['name']} v{nlp.meta['version']}")

Loaded pipeline: core_web_sm v3.9.4


## Working with Latin text

We will work with the opening sentences of Ritchie's *Fabulae Faciles* (1884), a classic Latin reader whose texts have long been standard in beginning Latin pedagogy. The passage introduces the myth of Perseus:

> *Haec narrantur a poetis de Perseo. Perseus filius erat Iovis, maximi deorum. Avus eius Acrisius appellabatur.*

Passing the text to `nlp()` runs it through the full LatinCy pipeline: tokenization, lemmatization, part-of-speech tagging, morphological analysis, dependency parsing, and named entity recognition.

In [3]:
text = (
    "Haec narrantur a poetis de Perseo. "
    "Perseus filius erat Iovis, maximi deorum. "
    "Avus eius Acrisius appellabatur."
)

doc = nlp(text)
print(doc)

Haec narrantur a poetis de Perseo. Perseus filius erat Iouis, maximi deorum. Auus eius Acrisius appellabatur.


## Token annotations

Once a text is processed, every token in the `Doc` carries a set of linguistic annotations. The most commonly used are:

| Attribute | Description |
|-----------|-------------|
| `token.text` | The surface form (after any pipeline normalization) |
| `token.lemma_` | The dictionary headword |
| `token.pos_` | Coarse part-of-speech tag (`NOUN`, `VERB`, `ADP`, …) |
| `token.morph` | Full morphological feature bundle |

LatinCy's normalization component converts *v* → *u* and *j* → *i* directly in `token.text`, so the input *Iovis* is stored as *Iouis* and *Avus* as *Auus* — visible in the Token column below:

In [4]:
print(f"{'Token':<14} {'Lemma':<14} {'POS':<8} Morphology")
print("-" * 64)
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f"{token.text:<14} {token.lemma_:<14} {token.pos_:<8} {token.morph}")

Token          Lemma          POS      Morphology
----------------------------------------------------------------
Haec           hic            DET      Case=Nom|Gender=Neut|Number=Plur
narrantur      narro          VERB     Aspect=Imp|Mood=Ind|Number=Plur|Person=3|Tense=Pres|VerbForm=Fin|Voice=Pass
a              ab             ADP      
poetis         poeta          NOUN     Case=Abl|Gender=Fem|Number=Plur
de             de             ADP      
Perseo         Perseus        PROPN    Case=Abl|Gender=Masc|Number=Sing
Perseus        Perseus        PROPN    Case=Nom|Gender=Masc|Number=Sing
filius         filius         NOUN     Case=Nom|Gender=Masc|Number=Sing
erat           sum            AUX      Aspect=Imp|Mood=Ind|Number=Sing|Person=3|Tense=Past|VerbForm=Fin
Iouis          Iuppiter       PROPN    Case=Gen|Gender=Masc|Number=Sing
maximi         magnus         ADJ      Case=Gen|Gender=Masc|Number=Sing
deorum         deus           NOUN     Case=Gen|Gender=Masc|Number=Plur
Auus       

## Finding words with the Matcher

Latin inflection means the same word can appear in many surface forms — *Perseus* in the nominative becomes *Perseo* in the ablative, *Perseum* in the accusative. A lemma-based search finds all of them at once.

spaCy's `Matcher` lets you search by annotation attributes. Matching on `LEMMA` finds every surface form the model has traced back to a given headword:

In [5]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
matcher.add('PERSEUS', [[{'LEMMA': 'Perseus'}]])

matches = matcher(doc)
print(f"{'Form':<12} {'POS':<8} Morphology")
print("-" * 52)
for match_id, start, end in matches:
    token = doc[start]
    print(f"{token.text:<12} {token.pos_:<8} {token.morph}")

Form         POS      Morphology
----------------------------------------------------
Perseo       PROPN    Case=Abl|Gender=Masc|Number=Sing
Perseus      PROPN    Case=Nom|Gender=Masc|Number=Sing


The same approach works for any set of terms. Searching for three key words from the passage — the hero, his divine father, and the family relationship — maps each match back to its lemma:

In [6]:
matcher = Matcher(nlp.vocab)
key_terms = ['Perseus', 'deus', 'filius']
for term in key_terms:
    matcher.add(term.upper(), [[{'LEMMA': term}]])

matches = matcher(doc)
print(f"{'Lemma':<10} {'Form':<14} Morphology")
print("-" * 56)
for match_id, start, end in matches:
    token = doc[start]
    print(f"{token.lemma_:<10} {token.text:<14} {token.morph}")

Lemma      Form           Morphology
--------------------------------------------------------
Perseus    Perseo         Case=Abl|Gender=Masc|Number=Sing
Perseus    Perseus        Case=Nom|Gender=Masc|Number=Sing
filius     filius         Case=Nom|Gender=Masc|Number=Sing
deus       deorum         Case=Gen|Gender=Masc|Number=Plur


## Next steps

This quickstart covers the basics. The full book works through each pipeline component in detail:

- **[Installing LatinCy models](2_install.ipynb)** — all four model sizes and optional packages
- **[Key annotations](4_key-annotations.ipynb)** — complete reference for token, span, and doc attributes
- **[Lemmatization](lemmatization.ipynb)** — how the lemmatizer works and its edge cases
- **[Sequence Matching](matcher.ipynb)** — the full `Matcher` API with operators, quantifiers, and regex patterns
- **[Named Entity Recognition](ner.ipynb)** — finding people, places, and other entities in Latin texts